# 📘 Notebook 18: Trees & Binary Search Trees

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module D: Data Structures · Notebook 4 of 4 in this module · (18 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Describe a **tree** using the correct vocabulary: root, parent, child, leaf, subtree, height
- Build a **binary tree** from nodes and process it with recursion
- Visit every node with in-order, pre-order, post-order and level-order **traversals**
- State the **binary search tree** property and explain why it makes searching fast
- Implement insertion and search in a binary search tree
- Explain what happens when a tree becomes **unbalanced**
- Choose between all the data structures of this course for a given task

> **Prerequisites:** Notebooks 12 (recursion), 13 (binary search), 15 (queues) and 16 (linked lists).
>
> 🔭 **Where everything meets.** This final notebook draws on almost every idea of the course: nodes joined by references (linked lists), halving the problem at every step (binary search), functions that call themselves (recursion), and a queue for one of the traversals. A hash table gave us O(1) lookup but no order. A sorted list gave us order but slow insertion. The binary search tree offers both order **and** speed.

## 1. From chains to trees

### Intuition first
In a linked list every node has **one** successor, so the nodes form a chain. Allow each node to have **several** successors and the chain becomes a **tree**: a structure that branches.

Trees are everywhere. The folders on your computer form a tree: a folder contains files and other folders. So do a family tree, the chapters and sections of a book, and the management chart of a company. Whenever data is **hierarchical**, a tree is its natural shape.

### Vocabulary
Computer scientists draw trees upside down, with the root at the top.

```
            8              <- root
          /   \
         3     10          <- children of 8
        / \      \
       1   6      14       <- 1 is a leaf
          / \
         4   7             <- leaves
```

| Term | Meaning |
|---|---|
| **root** | the single node at the top, with no parent |
| **child** | a node directly below another |
| **parent** | the node directly above |
| **leaf** | a node with no children |
| **subtree** | a node together with everything below it. It is itself a tree |
| **height** | the number of levels: the tree above has height 4 |

### Formal definition
A **binary tree** is a tree in which every node has **at most two** children, called the **left** child and the **right** child.

The definition of a subtree contains the key insight of this notebook: **a tree is a node plus two smaller trees.** A structure defined in terms of smaller copies of itself is a *recursive* structure, and recursion is the natural way to process it.

## 2. Building a tree by hand

A tree node is a linked-list node with two references instead of one.

In [ ]:
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

# Build the tree drawn above, node by node
root = TreeNode(8)
root.left = TreeNode(3)
root.right = TreeNode(10)
root.left.left = TreeNode(1)
root.left.right = TreeNode(6)
root.left.right.left = TreeNode(4)
root.left.right.right = TreeNode(7)
root.right.right = TreeNode(14)

print(root.value)
print(root.left.value, root.right.value)
print(root.left.right.left.value)

### Displaying a tree
To see what we build, here is a small function that prints a tree **turned on its side**: the root is on the left, and the right subtree is printed above the left one. Tilt your head to the left to read it. It is our first recursive function on a tree.

In [ ]:
def print_tree(node, level=0):
    if node is None:
        return
    print_tree(node.right, level + 1)
    print("      " * level + str(node.value))
    print_tree(node.left, level + 1)

print_tree(root)

## 3. Recursion on trees

Every question about a tree can be asked in the same way: *answer it for the left subtree, answer it for the right subtree, and combine the two answers with the node itself.* The base case is the empty tree, `None`.

### A worked example: counting the nodes
- Empty tree: 0 nodes.
- Otherwise: 1 (this node) + the nodes on the left + the nodes on the right.

### A worked example: the height
- Empty tree: height 0.
- Otherwise: 1 (this level) + the height of the **taller** subtree.

In [ ]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def height(node):
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

print("Nodes: ", count_nodes(root))
print("Height:", height(root))

> 🧠 Compare these with the linked-list functions of Notebook 16, which needed a loop and a `current` variable. Here there is no loop at all. Try writing `count_nodes` with a loop: it is surprisingly hard, because at every node there are two directions to follow.

## 4. Traversals: visiting every node

A list can be walked in one obvious order. A tree offers several, depending on **when** we handle a node relative to its subtrees.

| Traversal | Order | Result on our tree |
|---|---|---|
| **in-order** | left subtree, **node**, right subtree | 1 3 4 6 7 8 10 14 |
| **pre-order** | **node**, left subtree, right subtree | 8 3 1 6 4 7 10 14 |
| **post-order** | left subtree, right subtree, **node** | 1 4 7 6 3 14 10 8 |

The three functions differ only in the position of one line.

In [ ]:
def in_order(node):
    if node is None:
        return
    in_order(node.left)
    print(node.value, end=" ")
    in_order(node.right)

def pre_order(node):
    if node is None:
        return
    print(node.value, end=" ")
    pre_order(node.left)
    pre_order(node.right)

def post_order(node):
    if node is None:
        return
    post_order(node.left)
    post_order(node.right)
    print(node.value, end=" ")

print("In-order:   ", end="")
in_order(root)
print("\nPre-order:  ", end="")
pre_order(root)
print("\nPost-order: ", end="")
post_order(root)

Look at the in-order result: the values came out **in sorted order**. That is not a coincidence, and we will return to it in a moment.

### Level-order traversal
A fourth traversal visits the tree **level by level**, from top to bottom. It is not recursive. It uses a **queue** (Notebook 15): take a node from the front, handle it, and put its children at the back. Nodes are therefore handled in the order in which they were discovered.

In [ ]:
from collections import deque

def level_order(root):
    queue = deque([root])
    while queue:
        node = queue.popleft()
        print(node.value, end=" ")
        if node.left is not None:
            queue.append(node.left)
        if node.right is not None:
            queue.append(node.right)

level_order(root)

> 🧠 The three recursive traversals go **deep** before going wide, and they rely on the call **stack**. Level-order goes **wide** before going deep, and it relies on a **queue**. Stack and queue, depth and breadth: the two structures of Notebook 15 correspond to the two fundamental ways of exploring.

## 5. The binary search tree

### Intuition first
The tree we built by hand was not arbitrary. Look at the root, 8. **Everything** in its left subtree (1, 3, 4, 6, 7) is smaller than 8, and everything in its right subtree (10, 14) is larger. The same holds at node 3, at node 6, and at every other node.

### Formal definition
A **binary search tree** (BST) is a binary tree in which, for **every** node:

- all the values in its **left** subtree are **smaller** than the node's value;
- all the values in its **right** subtree are **larger**.

### Why it matters
To find a value we start at the root and compare. If the target is smaller, it can only be in the left subtree, so we **discard the entire right subtree**. If it is larger, we discard the left. One comparison removes a whole branch.

This is binary search (Notebook 13), built into the shape of the data. And unlike a sorted list, inserting a new value does not require shifting anything: we just attach a new leaf.

### Searching
```
Looking for 7:
  8  -> 7 is smaller, go left
  3  -> 7 is larger,  go right
  6  -> 7 is larger,  go right
  7  -> found
```

In [ ]:
def search(node, target):
    if node is None:
        return False                          # fell off the tree: not present
    if target == node.value:
        return True
    elif target < node.value:
        return search(node.left, target)      # only the left subtree can contain it
    else:
        return search(node.right, target)     # only the right subtree can contain it

print(search(root, 7))
print(search(root, 5))

### Inserting
Insertion follows the same path as a search. We walk down, going left or right, until we reach an empty spot, and place the new node there. The function **returns the root of the subtree** it was given, which lets each level re-attach the result. We ignore duplicates.

In [ ]:
def insert(node, value):
    if node is None:
        return TreeNode(value)                # empty spot found: the new node goes here
    if value < node.value:
        node.left = insert(node.left, value)
    elif value > node.value:
        node.right = insert(node.right, value)
    return node

# Build a BST by inserting values one at a time
root = None
for value in [50, 30, 70, 20, 40, 60, 80]:
    root = insert(root, value)

print_tree(root)

The first value inserted becomes the root. Every later value finds its own place by comparison alone. Adding one more:

In [ ]:
root = insert(root, 65)
print_tree(root)
print()
in_order(root)

### In-order traversal of a BST gives sorted output
"Left subtree, node, right subtree" means "everything smaller, then the node, then everything larger". Applied at every level, that is exactly ascending order. A BST therefore keeps its data **permanently sorted**, and reading it out in order costs only O(n).

## 6. A `BinarySearchTree` class

As with the linked list, we wrap the functions in a class that owns the root and offers a clean interface.

In [ ]:
class BinarySearchTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def insert(self, value):
        if not self.contains(value):
            self.size += 1
        self.root = insert(self.root, value)

    def contains(self, value):
        return search(self.root, value)

    def to_sorted_list(self):
        result = []
        self._collect(self.root, result)
        return result

    def _collect(self, node, result):
        if node is None:
            return
        self._collect(node.left, result)
        result.append(node.value)
        self._collect(node.right, result)

    def minimum(self):
        node = self.root
        while node.left is not None:        # the smallest value is as far left as possible
            node = node.left
        return node.value

    def maximum(self):
        node = self.root
        while node.right is not None:       # the largest value is as far right as possible
            node = node.right
        return node.value

tree = BinarySearchTree()
for value in [50, 30, 70, 20, 40, 60, 80]:
    tree.insert(value)

print("Contains 60?", tree.contains(60))
print("Contains 65?", tree.contains(65))
print("Sorted:", tree.to_sorted_list())
print("Minimum:", tree.minimum(), "| Maximum:", tree.maximum())
print("Size:", tree.size)

## 7. How fast is it?

A search follows **one path** from the root downwards, so it makes at most as many comparisons as the tree has levels. The cost of a search is the **height** of the tree.

How tall is a tree with `n` nodes? If it is well filled, each level holds twice as many nodes as the one above: 1, 2, 4, 8, ... A tree of height 20 has room for more than a million nodes. So the height is about **log n**, and search and insertion are both **O(log n)**.

### Experiment: the height of a tree built from random values

In [ ]:
import random
import math

print(f"{'n':>8} {'height':>8} {'log2(n)':>9}")
for n in [100, 1_000, 10_000, 100_000]:
    values = random.sample(range(n * 10), n)      # n distinct random numbers
    root = None
    for value in values:
        root = insert(root, value)
    print(f"{n:>8,} {height(root):>8} {math.log2(n):>9.1f}")

The tree grew a thousand times larger and its height did not even triple. A tree built from random values is not perfectly filled, so its height is two or three times log₂(n), but it grows at the same slow rate. A hundred thousand values can be searched in a few dozen comparisons.

### Experiment: BST against a plain list

In [ ]:
import time

n = 20_000
values = random.sample(range(n * 10), n)
targets = random.sample(range(n * 10), 1_000)

as_list = list(values)
root = None
for value in values:
    root = insert(root, value)

start = time.perf_counter()
for target in targets:
    target in as_list
list_time = time.perf_counter() - start

start = time.perf_counter()
for target in targets:
    search(root, target)
tree_time = time.perf_counter() - start

print(f"List (linear search): {list_time:.4f} seconds")
print(f"Binary search tree:   {tree_time:.4f} seconds")

## 8. The weakness: unbalanced trees

### The problem
Everything above assumed a "well filled" tree. The shape of a BST depends on **the order in which the values are inserted**. What if they arrive already sorted?

In [ ]:
root = None
for value in [10, 20, 30, 40, 50]:
    root = insert(root, value)

print_tree(root)
print("Height:", height(root))

Each value is larger than all the previous ones, so each one goes to the right of the last. The "tree" has **no branches**. It has degenerated into a linked list, its height is `n`, and searching it is **O(n)**. All the advantage is lost.

In [ ]:
n = 500

random_values = random.sample(range(n), n)
root_random = None
for value in random_values:
    root_random = insert(root_random, value)

root_sorted = None
for value in range(n):
    root_sorted = insert(root_sorted, value)

print(f"{n} values in random order: height {height(root_random)}")
print(f"{n} values in sorted order: height {height(root_sorted)}")

> ⚠️ **Common pitfalls**
>
> - Building a BST from data that is already sorted. It is the worst possible input, and a very common one in practice.
> - A degenerate tree is also as **deep** as it is large. Recursive functions on it can exceed Python's limit of about 1,000 nested calls, which is why this experiment uses only 500 values.

### The solution: self-balancing trees
The remedy is a tree that **rearranges itself** during insertion, so that its height always stays close to log n. The best-known kinds are **AVL trees** and **red-black trees**. They guarantee O(log n) for search, insertion and deletion, whatever the order of the input. They are the engines behind database indexes and the ordered containers of many programming languages. Their details belong to a more advanced course, but you now understand exactly the problem they solve.

## 9. The whole course in one table

You have now built every major linear and hierarchical structure by hand. Here they are side by side. "Search" means finding a value; "ordered" means the structure can give you its elements in sorted order, or the smallest, or the next larger.

| Structure | Access by index | Search | Insert | Ordered? | Choose it when... |
|---|---|---|---|---|---|
| Python list (array) | **O(1)** | O(n) | O(1) at the end, O(n) at the front | no | you need positions and mostly append |
| Sorted list | **O(1)** | **O(log n)** | O(n) | **yes** | the data rarely changes and is searched often |
| Stack | top only | - | **O(1)** | no | the most recent item comes first |
| Queue | front only | - | **O(1)** | no | items are handled in order of arrival |
| Linked list | O(n) | O(n) | **O(1)** at the front | no | you insert and remove at the ends |
| Hash table | - | **O(1)** | **O(1)** | no | you look things up by key |
| Binary search tree | - | **O(log n)** | **O(log n)** | **yes** | you need fast lookup **and** order |

The BST figures assume a balanced tree. The hash table figures are averages.

There is no row that wins every column. **Choosing a data structure means deciding which operations matter most**, and that decision often matters more than anything else in a program.

---
## ✏️ Exercises

Unless stated otherwise, the functions `insert`, `search`, `height`, `in_order` and `print_tree` and the class `TreeNode` are available from the cells above. Start each exercise by building this tree:

```python
root = None
for value in [50, 30, 70, 20, 40, 60, 80, 35, 45]:
    root = insert(root, value)
```

### Exercise 1 (Draw it first)
On paper, draw the BST produced by inserting `50, 30, 70, 20, 40, 60, 80, 35, 45` in that order. Then check with `print_tree`. What is its height?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
root = None
for value in [50, 30, 70, 20, 40, 60, 80, 35, 45]:
    root = insert(root, value)

print_tree(root)
print("Height:", height(root))
```
</details>

### Exercise 2 (Sum of all values)
Write a recursive function `tree_sum(node)` that returns the sum of all the values in a tree.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def tree_sum(node):
    if node is None:
        return 0
    return node.value + tree_sum(node.left) + tree_sum(node.right)

print(tree_sum(root))
```
</details>

### Exercise 3 (Count the leaves)
Write a recursive function `count_leaves(node)` that returns the number of leaves, the nodes with no children.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def count_leaves(node):
    if node is None:
        return 0
    if node.left is None and node.right is None:
        return 1                                   # this node is a leaf
    return count_leaves(node.left) + count_leaves(node.right)

print(count_leaves(root))
```
</details>

### Exercise 4 (Count the comparisons)
Write a function `search_counted(node, target)` that returns the **number of nodes examined** while searching for a value. Use it to find how many comparisons are needed for 45, for 50 and for 99.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def search_counted(node, target):
    comparisons = 0
    while node is not None:
        comparisons += 1
        if target == node.value:
            break
        elif target < node.value:
            node = node.left
        else:
            node = node.right
    return comparisons

for target in [45, 50, 99]:
    print(target, search_counted(root, target))
```

*This version uses a loop instead of recursion. A search follows a single path, so a loop with one moving reference is enough, exactly as in a linked list.*
</details>

### Exercise 5 (Tree sort)
Write a function `tree_sort(values)` that sorts a list by inserting all its values into a BST and reading them back with an in-order traversal. Test it on 15 random numbers. What is its Big O for random input?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
import random

def tree_sort(values):
    root = None
    for value in values:
        root = insert(root, value)

    result = []

    def collect(node):
        if node is None:
            return
        collect(node.left)
        result.append(node.value)
        collect(node.right)

    collect(root)
    return result

data = random.sample(range(100), 15)
print(data)
print(tree_sort(data))
```

*`n` insertions at O(log n) each, plus an O(n) traversal: O(n log n), the same class as merge sort. On input that is already sorted it degrades to O(n²), for the reason shown in section 8. Note also that this `insert` ignores duplicates.*
</details>

### Exercise 6 (Print the levels)
Modify the level-order traversal so that each level is printed on **its own line**. For the tree of these exercises the first three lines are `50`, then `30 70`, then `20 40 60 80`. (Hint: at the start of each round, the queue holds exactly the nodes of one level. Record its length and process that many nodes.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
from collections import deque

def print_levels(root):
    queue = deque([root])
    while queue:
        level_size = len(queue)
        for i in range(level_size):
            node = queue.popleft()
            print(node.value, end=" ")
            if node.left is not None:
                queue.append(node.left)
            if node.right is not None:
                queue.append(node.right)
        print()

print_levels(root)
```
</details>

### Exercise 7 (Values in a range)
Write a function `in_range(node, low, high)` that returns a sorted list of the values between `low` and `high` (inclusive). Use the BST property to **avoid visiting subtrees that cannot contain any such value**.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def in_range(node, low, high):
    if node is None:
        return []
    result = []
    if node.value > low:                       # the left subtree may hold values >= low
        result = result + in_range(node.left, low, high)
    if low <= node.value <= high:
        result.append(node.value)
    if node.value < high:                      # the right subtree may hold values <= high
        result = result + in_range(node.right, low, high)
    return result

print(in_range(root, 35, 60))
```

*This is a question a hash table cannot answer without looking at every key. Ordered structures exist for exactly this kind of query.*
</details>

### Exercise 8 (Is it a valid BST? A little harder)
Write a function `is_bst(node, low, high)` that checks whether a binary tree satisfies the BST property. It is **not** enough to compare each node with its two children: every value in the left subtree must be smaller than the node, however deep it lies. (Hint: pass down the range of values allowed for each subtree. Start with `is_bst(root, float("-inf"), float("inf"))`, which means "no limits".)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def is_bst(node, low, high):
    if node is None:
        return True
    if not (low < node.value < high):
        return False
    return (is_bst(node.left, low, node.value) and        # left: must stay below this node
            is_bst(node.right, node.value, high))         # right: must stay above this node

print(is_bst(root, float("-inf"), float("inf")))

# A tree that fools the naive check: 6 is in the left subtree of 5
bad = TreeNode(5)
bad.left = TreeNode(3)
bad.left.right = TreeNode(6)
print(is_bst(bad, float("-inf"), float("inf")))
```

*In the bad tree every node looks fine next to its own children (6 is larger than 3), yet 6 sits in the left subtree of 5. Passing the allowed range downwards catches it.*
</details>

## 🔑 Key takeaways

- A **tree** is a hierarchical structure of nodes: root, parents, children, leaves. A **binary tree** allows at most two children per node.
- A tree is a node plus two smaller trees, so tree algorithms are naturally **recursive**, with `None` as the base case.
- **In-order**, **pre-order** and **post-order** traversals go deep first; **level-order** uses a queue and goes wide first.
- In a **binary search tree**, everything on the left of a node is smaller and everything on the right is larger.
- Search and insertion follow a single path: **O(height)**, which is **O(log n)** for a balanced tree.
- An in-order traversal of a BST yields the values **in sorted order**.
- Sorted input produces a degenerate tree of height `n`. **Self-balancing trees** exist to prevent this.
- No data structure wins everywhere. Choosing one means deciding which operations matter most.

### 🎓 Congratulations: you have completed the entire course!

You began by printing *Hello, World!* and you finish by building a binary search tree and reasoning about its height. Along the way you learned the Python language, wrote complete programs from a design, measured how algorithms scale, and constructed stacks, queues, linked lists, hash tables and trees **by hand**, before trusting the built-in versions.

At the start we said that one question would run through the whole course: *"how many steps does this take?"* You can now ask it of any program, answer it by reading the code, and confirm the answer with an experiment. That habit, more than any single algorithm, is what you take with you.

From here the road leads to balanced trees, heaps, graphs and the algorithms built on them. You have the foundations for all of it.

Thank you for working through all eighteen notebooks.

---
*End of Module D: and the end of the course.*

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*